# Exploração inicial — Dataset Olist

Objetivo: entender a estrutura dos dados ANTES de escrever o ETL.
Para cada tabela, vamos olhar: as primeiras linhas, os tipos de coluna,
os valores nulos e os valores únicos de colunas-chave.

> Nota: este notebook roda sobre o dataset sintético gerado em
> `data/generate_sample_data.py` (mesmo formato do dataset real da Olist).
> Para usar os dados reais, baixe-os no Kaggle e substitua os arquivos em
> `data/raw/` — nada mais precisa mudar.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

## 1. Tabela `orders`

In [2]:
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,ord_00000006968,cust_00016969094,delivered,2017-09-04 18:32:00,2017-09-04 20:32:00,2017-09-06 20:32:00,2017-09-10 20:32:00,2017-09-15 18:32:00
1,ord_00000018577,cust_00002434111,delivered,2018-03-29 12:43:00,2018-03-29 17:43:00,2018-04-01 17:43:00,2018-04-07 17:43:00,2018-04-07 12:43:00
2,ord_00000023420,cust_00019104397,delivered,2018-04-25 14:00:00,2018-04-27 03:00:00,2018-04-28 03:00:00,2018-05-07 03:00:00,2018-05-03 14:00:00
3,ord_00000034938,cust_00014218967,delivered,2018-04-12 15:41:00,2018-04-13 18:41:00,2018-04-15 18:41:00,2018-04-18 18:41:00,2018-04-24 15:41:00
4,ord_00000043465,cust_00012749149,delivered,2018-03-17 21:25:00,2018-03-18 23:25:00,2018-03-21 23:25:00,2018-03-26 23:25:00,2018-03-30 21:25:00


In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 5200 entries, 0 to 5199
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       5200 non-null   str  
 1   customer_id                    5200 non-null   str  
 2   order_status                   5200 non-null   str  
 3   order_purchase_timestamp       5200 non-null   str  
 4   order_approved_at              5158 non-null   str  
 5   order_delivered_carrier_date   5200 non-null   str  
 6   order_delivered_customer_date  4871 non-null   str  
 7   order_estimated_delivery_date  5200 non-null   str  
dtypes: str(8)
memory usage: 325.1 KB


In [4]:
orders.isnull().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 42
order_delivered_carrier_date       0
order_delivered_customer_date    329
order_estimated_delivery_date      0
dtype: int64

In [5]:
orders['order_status'].value_counts()

order_status
delivered      4871
shipped         167
canceled         85
processing       50
unavailable      27
Name: count, dtype: int64

**O que observamos aqui:**
- As colunas de data (`order_purchase_timestamp`, `order_approved_at` etc.) aparecem
  como `object` (texto), não como `datetime` — precisa converter no ETL.
- `order_approved_at` tem alguns nulos.
- A grande maioria dos pedidos está com status `delivered`.

## 2. Tabela `customers`

In [6]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,cust_00000002824,uniq_00000009872,18266.0,São Paulo,SP
1,cust_00000011409,uniq_00000016675,87328.0,Campo Grande,MS
2,cust_00000025506,uniq_00000021157,86988.0,Curitiba,PR
3,cust_00000035012,uniq_00000036488,44526.0,Belo Horizonte,MG
4,cust_00000044657,uniq_00000041023,12889.0,São Paulo,SP


In [7]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3015 entries, 0 to 3014
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               3015 non-null   str    
 1   customer_unique_id        3015 non-null   str    
 2   customer_zip_code_prefix  2985 non-null   float64
 3   customer_city             3015 non-null   str    
 4   customer_state            3015 non-null   str    
dtypes: float64(1), str(4)
memory usage: 117.9 KB


In [8]:
customers.isnull().sum()

customer_id                  0
customer_unique_id           0
customer_zip_code_prefix    30
customer_city                0
customer_state               0
dtype: int64

In [9]:
customers['customer_state'].value_counts().head(10)

customer_state
SP    1247
RJ     379
MG     323
RS     168
PR     140
BA     115
SC     105
ES      79
PE      60
DF      54
Name: count, dtype: int64

**O que observamos aqui:**
- `customer_zip_code_prefix` tem alguns nulos.
- SP concentra a maior parte dos clientes, seguido de RJ e MG — condizente com a
  distribuição populacional/econômica do Brasil.

## 3. Tabela `order_items`

In [10]:
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
items.head()

,order_id,order_item_id,product_id,seller_id,price,freight_value
0,ord_00000006968,1,prod_00001058504,sell_00002213440,166.66,14.02
1,ord_00000018577,1,prod_00003639425,sell_00001147861,165.15,14.71
2,ord_00000018577,2,prod_00004899301,sell_00002538598,112.80,19.78
3,ord_00000023420,1,prod_00004788056,sell_00002694393,382.47,21.39
4,ord_00000023420,2,prod_00002629489,sell_00000711746,196.69,24.22


In [11]:
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 8723 entries, 0 to 8722
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       8723 non-null   str    
 1   order_item_id  8723 non-null   int64  
 2   product_id     8723 non-null   str    
 3   seller_id      8723 non-null   str    
 4   price          8723 non-null   float64
 5   freight_value  8723 non-null   float64
dtypes: float64(2), int64(1), str(3)
memory usage: 409.0 KB


In [12]:
items.isnull().sum()

order_id         0
order_item_id    0
product_id       0
seller_id        0
price            0
freight_value    0
dtype: int64

In [13]:
items[['price', 'freight_value']].describe()

,price,freight_value
count,8723.000000,8723.000000
mean,169.528084,18.191410
std,113.495834,6.710285
min,9.900000,6.500000
25%,74.510000,13.240000
50%,149.800000,18.020000
75%,251.615000,22.880000
max,628.910000,41.360000


**O que observamos aqui:**
- Sem nulos relevantes nessa tabela.
- `price` e `freight_value` têm uma distribuição bem espalhada — vale conferir
  outliers antes de tirar conclusões de negócio (ex.: produtos de frete muito caro).

## 4. Checklist para o ETL (roteiro do próximo passo)

- [x] Converter colunas de data em `orders` e `order_reviews` para `datetime`
- [x] Remover duplicatas em `customers`, `products`, `sellers`, `order_reviews`
- [x] Preencher `product_category_name` nulo com `'nao_informado'`
- [x] Selecionar apenas as colunas usadas no schema (`products`, `order_reviews`)
- [x] Garantir integridade referencial antes do `load` (só inserir linhas cujas FKs existem)

Esse checklist virou exatamente o `src/etl.py` deste projeto.